In [280]:
import pandas as pd

df = pd.read_csv('./data/logistics_dirty_dataset.csv')

In [281]:
""" Drop duplicate rows from data frame"""
df = df.drop_duplicates()
df.shape

(150000, 20)

## Dropping the Notes and Remarks columns

In [282]:
""" I am dropping the Notes and Remarks columns bcos they don't really have a major impact on the data"""
df = df.drop(labels=['Notes', 'Remarks'], axis=1)


## Fixing the nulls in the Shipment_ID

In [ ]:
""" Fixing the nulls in the Shipment_ID column by generating new set of ID for nulls"""
# replace_IDs = [f'SHPFILL{num}' for num in range(1, 1475, 1)]

null_count = df['Shipment_ID'].isnull().sum()
replace_IDs = [f'SHPFILL{num}' for num in range(1, null_count + 1)]
mask = df['Shipment_ID'].isnull()
df.loc[mask, 'Shipment_ID'] = replace_IDs

df['Shipment_ID'].value_counts()

Shipment_ID
SHP58805      10
SHP59414       9
SHP36775       9
SHP53133       9
SHP49235       9
              ..
SHP15658       1
SHPFILL696     1
SHP76989       1
SHP64802       1
SHP33043       1
Name: count, Length: 74038, dtype: int64

## Standardizing all categorical columns

In [302]:
""" Standardizing all categorical columns, fixing whitespace, upper/lower case non-uniformity and typo error"""

category_columns = ['Payment_Method', 'Insurance', 'Destination', 'Origin_City', 'Priority', 'Vehicle_Type', 'Status']
for column in category_columns:
    df[f'{column}'] = df[f'{column}'].str.strip().str.title()
    print(df[f'{column}'].unique())



df['Payment_Method'] = df['Payment_Method'].replace({'Tranfer': 'Transfer'})
df['Vehicle_Type'] = df['Vehicle_Type'].replace({'Bke': 'Bike'})
df['Status'] = df['Status'].replace({'Delayd': 'Delayed'})

print()
print(f'Payment_Method: {df['Payment_Method'].unique()} \n Vehicle_Type: {df['Vehicle_Type'].unique()} \n Status: {df['Status'].unique()} ')

['Cash' 'Card' 'Transfer']
['No' 'Yes']
['Customer Hub' 'Retail Store']
['Abuja' 'Lagos' 'Port Harcourt' 'Kano']
['High' 'Medium' 'Low']
['Van' 'Bike' 'Truck']
['Delivered' 'In Transit' 'Cancelled' 'Delayed']

Payment_Method: ['Cash' 'Card' 'Transfer'] 
 Vehicle_Type: ['Van' 'Bike' 'Truck'] 
 Status: ['Delivered' 'In Transit' 'Cancelled' 'Delayed'] 


## Fixing the date columns

In [296]:
""" Fixing the date columns"""
df['Shipment_Date'] = pd.to_datetime(df['Shipment_Date'], errors='coerce')
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'], errors='coerce')

print(f'Shipment_Date has: {df['Shipment_Date'].isnull().sum()} null values \n Delivery_Date has: {df['Delivery_Date'].isnull().sum()} null values')

""" I decided to drop the NaT values because a logistics dataset is meant to have a valid Shipment_Date and Delivery_Date"""

df = df.dropna(subset=['Shipment_Date', 'Delivery_Date'])

print(f'Rows remaining after dropping invalid dates: {df.shape[0]:,}')

Shipment_Date has: 0 null values 
 Delivery_Date has: 0 null values
Rows remaining after dropping invalid dates: 144,036


## Handling Negative values

In [297]:
""" Handling Negative values """
""" I convert the negative values to NaN first"""
""" I then filled up those NaN with the median of each column"""

cols_negative = ['Distance_km', 'Delivery_Days', 'Cost_NGN', 'Weight_kg']
for column in cols_negative:
    df.loc[df[column] < 0, column] = pd.NA
    median_val = df[column].median()
    df[column] = df[column].fillna(median_val)


## More Fixes

In [298]:
""" Fixing the data type of Delivery_Days as wellas the Nulls + datatype of Customer_Rating"""

df['Delivery_Days'] = df['Delivery_Days'].astype(int)

median_rating = df[ 'Customer_Rating'].median()
df['Customer_Rating'] = df['Customer_Rating'].fillna(median_rating)
df['Customer_Rating'] = df['Customer_Rating'].astype(int)


## Delivery_Duration mismatch check

In [299]:
""" Here i created a new column more like a validation column 'Delivery_Duration' 
which calculates the actual number of days between shipment and delivery"""

df['Delivery_Duration'] = (df['Delivery_Date'] - df['Shipment_Date']).dt.days
df['Delivery_Duration'] = df['Delivery_Duration'].astype(int)
mismatch_check = (df['Delivery_Duration'] != df['Delivery_Days']).sum()

print(f'Rows where Delivery_Duration is not same as Delivery_Days: {mismatch_check:,}')



Rows where Delivery_Duration is not same as Delivery_Days: 2,802


## FINAL VALIDATION

In [300]:
""" FINAL VALIDATION """

df.duplicated().sum()
df.isnull().sum()

rows, cols = df.shape
print(f'A total of {(150500 - rows):,} rows were lost from the original 150,500')

category_cols = ['Payment_Method', 'Insurance', 'Destination', 'Origin_City', 'Priority', 'Vehicle_Type', 'Status']
for column in category_cols:
    print(df[f'{column}'].unique())

print(f'Remaining duplicates: {df.duplicated().sum()}')
print()
print('Remaining nulls:')
print(df.isnull().sum())


A total of 6,464 rows were lost from the original 150,500
['Cash' 'Card' 'Transfer']
['No' 'Yes']
['Customer Hub' 'Retail Store']
['Abuja' 'Lagos' 'Port Harcourt' 'Kano']
['High' 'Medium' 'Low']
['Van' 'Bike' 'Truck']
['Delivered' 'In Transit' 'Cancelled' 'Delayed']
Remaining duplicates: 0

Remaining nulls:
Shipment_ID          0
Order_ID             0
Origin_City          0
Warehouse            0
Destination          0
Weight_kg            0
Cost_NGN             0
Distance_km          0
Delivery_Days        0
Shipment_Date        0
Delivery_Date        0
Status               0
Vehicle_Type         0
Payment_Method       0
Priority             0
Insurance            0
Driver_Name          0
Customer_Rating      0
Delivery_Duration    0
dtype: int64


## Exporting the Cleaned Dataset

In [303]:
df.to_csv('./data/logistics_CLEANED_data.csv', index=False)
print(f'Clean dataset exported: {df.shape[0]:,} rows, {df.shape[1]} columns')

Clean dataset exported: 144,036 rows, 19 columns
